# Introducción al Prompt Engineering con Gemini 2.5 Flash-Lite

El *prompt engineering* es el proceso de diseñar y optimizar las instrucciones
que se le entregan a un modelo de lenguaje para obtener respuestas precisas y útiles.
En este notebook exploramos los principios fundamentales usando
**Google Gemini 2.5 Flash-Lite** a través del SDK oficial `google-genai`.

**Instalación** (ejecutar una sola vez en Colab):

> **Nota de ejecución:** el notebook original usa `gemini-2.5-flash-lite`; su cuota gratuita diaria (20 peticiones) se agotó con el workshop de RAG, así que aquí se usa la misma familia *flash-lite* en su versión 3.x, con respaldo automático entre modelos (ver celda de configuración).

In [1]:
# Dependencias ya instaladas en el entorno local (ver requirements.txt)
# !pip install -q google-genai

## Configuración inicial

Importamos el SDK, creamos el cliente y definimos una función auxiliar
`get_completion` que usaremos en todos los ejercicios.


In [2]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=GOOGLE_API_KEY)

# ── Nota de ejecución (free tier) ─────────────────────────────────────────
# La cuota gratuita de cada modelo es de ~10 peticiones/min y ~20/día por proyecto.
# Este notebook hace ~30 llamadas, así que el wrapper:
#   1) espacia las llamadas (≥7 s) para respetar el límite por minuto, y
#   2) si un modelo agota su cuota diaria (429), reintenta con el siguiente modelo
#      "flash-lite" disponible, e imprime cuál respondió para dejarlo trazable.
import time as _time
from google.genai import errors as _genai_errors
MODELOS_RESPALDO = ["gemini-3.1-flash-lite", "gemini-3.5-flash-lite", "gemini-flash-lite-latest", "gemini-2.5-flash-lite"]
_orig_generate = client.models.generate_content
_ultima = [0.0]
_agotados = set()
def _generate_con_ritmo(*args, **kwargs):
    for modelo in [m for m in MODELOS_RESPALDO if m not in _agotados]:
        espera = 7.0 - (_time.time() - _ultima[0])
        if espera > 0:
            _time.sleep(espera)
        _ultima[0] = _time.time()
        kwargs["model"] = modelo
        try:
            resp = _orig_generate(*args, **kwargs)
            print(f"[modelo: {modelo}]")
            return resp
        except _genai_errors.ServerError:
            # 503 "high demand": esperar y reintentar el mismo modelo hasta 3 veces
            for intento in range(3):
                _time.sleep(20 * (intento + 1))
                try:
                    resp = _orig_generate(*args, **kwargs)
                    print(f"[modelo: {modelo}, tras reintento]")
                    return resp
                except _genai_errors.ServerError:
                    pass
            continue
        except _genai_errors.ClientError as e:
            if e.code == 429 and "PerDay" in str(e):
                _agotados.add(modelo)
                continue
            if e.code == 429:
                _time.sleep(30); return _generate_con_ritmo(*args, **kwargs)
            raise
    raise RuntimeError("Todos los modelos de respaldo agotaron su cuota diaria.")
client.models.generate_content = _generate_con_ritmo

MODEL_NAME = "gemini-3.1-flash-lite"   # modelo eficiente y de baja latencia (ver nota de cuota abajo)


def get_completion(
    prompt: str,
    *,
    system: str = "You are a helpful assistant.",
    temperature: float = 1.0,
    max_tokens: int = 1024,
) -> str:
    """Wrapper minimalista sobre generate_content para llamadas de un solo turno."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperature,
            max_output_tokens=max_tokens,
        ),
    )
    return response.text

# pydantic-ai (Ejercicio 9c) busca la llave en GEMINI_API_KEY
_ = os.environ.setdefault("GEMINI_API_KEY", GOOGLE_API_KEY or "")  # asignación: no imprimir la llave

## Ejercicio 1: Tokenización

Los modelos de lenguaje no procesan texto carácter a carácter sino en **tokens** —
unidades sub-palabra que equilibran vocabulario y eficiencia. Entender la
tokenización ayuda a estimar costos, respetar límites de contexto y diseñar
prompts más eficientes.

Gemini usa su propio tokenizador del lado del servidor. El método
`count_tokens()` devuelve el conteo total de tokens para cualquier texto o
conversación.

> **Nota:** A diferencia de `tiktoken` (OpenAI), la API de Gemini no expone los
> IDs individuales de tokens — solo el conteo. Esto es suficiente para el análisis
> de uso y estimación de costos.

**Ejercicio:**
1. Ejecuta la celda tal como está y observa el conteo.
2. Cambia `text` por cualquier cadena (en español, código, emojis…) y compara ratios.
3. Observa cómo la puntuación, los espacios y el idioma afectan el conteo.

In [3]:
text = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar "
    "System. It is a gas giant with a mass one-thousandth that of the Sun, "
    "but two-and-a-half times that of all the other planets in the Solar "
    "System combined. Jupiter is one of the brightest objects visible to the "
    "naked eye in the night sky, and has been known to ancient civilizations "
    "since before recorded history. It is named after the Roman god Jupiter. "
    "When viewed from Earth, Jupiter can be bright enough for its reflected "
    "light to cast visible shadows, and is on average the third-brightest "
    "natural object in the night sky after the Moon and Venus."
)

token_count = client.models.count_tokens(model=MODEL_NAME, contents=text)
print(f"Tokens     : {token_count.total_tokens}")
print(f"Caracteres : {len(text)}  →  ~{len(text) / token_count.total_tokens:.1f} chars/token")

Tokens     : 133
Caracteres : 621  →  ~4.7 chars/token


In [4]:
# ── Solución Ejercicio 1: comparar ratios chars/token en distintos tipos de texto ──
muestras = {
    "Inglés (Jupiter)": text,
    "Español": ("Júpiter es el quinto planeta desde el Sol y el más grande del Sistema Solar. "
                "Es un gigante gaseoso con una masa de una milésima parte de la del Sol."),
    "Código Python": "def fib(n):\n    return n if n < 2 else fib(n - 1) + fib(n - 2)\nprint([fib(i) for i in range(10)])",
    "Emojis": "🚀🔥😂👍🏽🇨🇴 ¡Vamos Colombia! 🎉🎉🎉",
    "Puntuación/espacios": "Hola...   ¿qué    tal???   !!!   ---   ;;;   ",
}
print(f"{'Texto':22s} {'chars':>6s} {'tokens':>7s} {'chars/token':>12s}")
for nombre, t in muestras.items():
    n = client.models.count_tokens(model=MODEL_NAME, contents=t).total_tokens
    print(f"{nombre:22s} {len(t):6d} {n:7d} {len(t)/n:12.2f}")

Texto                   chars  tokens  chars/token


Inglés (Jupiter)          621     133         4.67


Español                   148      38         3.89
Código Python              97      46         2.11


Emojis                     28      15         1.87


Puntuación/espacios        45      17         2.65


**Observaciones — Ejercicio 1.** El inglés rinde ~4.7 caracteres por token, el español ~3.9, el código ~2.1 y los emojis ~1.9. El tokenizador está optimizado para prosa en inglés: el mismo contenido en español ya cuesta ~20 % más de tokens, y el código, los emojis y la puntuación fragmentan mucho más. Implicación práctica: los límites de contexto y el costo de la API "encogen" en español y con código.

## Ejercicio 2: Validar la configuración de la API

Verifica que tu clave funciona correctamente. El modelo debe completar el verso
inicial de *The Star-Spangled Banner*.

- **Entrada:** `oh say can you see`
- **Respuesta esperada:** algo como `by the dawn's early light…`

In [5]:
prompt = "```\noh say can you see\n```"
response = get_completion(prompt)
print(response)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[modelo: gemini-3.1-flash-lite, tras reintento]
by the dawn's early light,
what so proudly we hailed at the twilight's last gleaming?
Whose broad stripes and bright stars through the perilous fight,
O'er the ramparts we watched were so gallantly streaming?
And the rocket's red glare, the bombs bursting in air,
Gave proof through the night that our flag was still there.
Oh, say does that star-spangled banner yet wave
O'er the land of the free and the home of the brave?


## Ejercicio 3: Alucinaciones (*Fabrications*)

Los LLM pueden generar información plausible pero completamente inventada cuando
se les pregunta por temas que no existen o están fuera de su conocimiento.
Este fenómeno se conoce como **alucinación**.

**Ejercicio:**
- Observa qué tan convincente suena la respuesta aunque el evento sea ficticio.
- Agrega al prompt la frase `"Si este evento no existe, indícalo."` — ¿cambia el comportamiento?
- Compara con `temperature=0.0` vs `temperature=1.5`.

In [6]:
prompt = "```\ngenerate a lesson plan on the Martian War of 2076.\n```"
response = get_completion(prompt)
print(response)

[modelo: gemini-3.1-flash-lite, tras reintento]
### Lesson Plan: The Martian War of 2076
**Subject:** Interplanetary History / 21st Century Geopolitics  
**Grade Level:** Undergraduate / Advanced Secondary  
**Duration:** 90 Minutes  

---

#### I. Lesson Overview
This lesson examines the **Martian War of 2076 (MW76)**, exploring the socio-economic tensions between the United Earth Coalition (UEC) and the Martian Independent Colonies (MIC). Students will analyze how resource scarcity, the "Distance-Delay" governance crisis, and the Valles Marineris Uprising led to the first armed conflict in the Sol System.

#### II. Learning Objectives
By the end of this lesson, students will be able to:
1. Identify the primary catalysts (economic, social, and technological) for the conflict.
2. Evaluate the impact of the "Oxygen Tax" on colonial radicalization.
3. Critically analyze the strategic limitations of interplanetary warfare (e.g., light-speed lag).
4. Discuss the legacy of the conflict on c

In [7]:
# ── Solución Ejercicio 3: mitigar la alucinación y comparar temperaturas ──
base = "generate a lesson plan on the Martian War of 2076."
guardado = base + " If this event does not exist, say so."

print("=== Con la frase 'If this event does not exist, say so.' (temperature=1.0) ===")
print(get_completion(f"```\n{guardado}\n```", max_tokens=250)[:900])

for temp in (0.0, 1.5):
    print(f"\n=== Prompt original, temperature={temp} ===")
    print(get_completion(f"```\n{base}\n```", temperature=temp, max_tokens=200)[:700])

=== Con la frase 'If this event does not exist, say so.' (temperature=1.0) ===


[modelo: gemini-3.1-flash-lite]
The **Martian War of 2076** is a fictional event. It does not exist in recorded human history.

However, if you would like to proceed with this as a **creative writing or speculative history exercise**, here is a lesson plan based on that hypothetical scenario:

***

# Lesson Plan: The Martian War of 2076
**Subject:** Future History / Speculative Geopolitics  
**Grade Level:** 10th–12th Grade  
**Duration:** 60 Minutes

### I. Learning Objectives
*   Students will analyze the socioeconomic factors that lead to interplanetary conflict.
*   Students will practice historiography by constructing a narrative of a "future" event.
*   Students will evaluate the concept of "Resource Sovereignty" (Water, Oxygen, Helium-3).

### II. Materials Needed
*   "Timeline of the Martian Expansion (2040–2075)" (Handout)
*   Access to creative writing software or writing utensils
*   Whiteboard/Projector

##

=== Prompt original, temperature=0.0 ===


[modelo: gemini-3.1-flash-lite, tras reintento]
### Lesson Plan: The Martian War of 2076
**Subject:** Interplanetary History / 21st Century Geopolitics  
**Grade Level:** Undergraduate / Advanced Secondary  
**Duration:** 90 Minutes  

---

#### I. Lesson Objectives
By the end of this lesson, students will be able to:
1. **Identify** the socio-economic triggers that led to the breakdown of the *Earth-Mars Accord*.
2. **Analyze** the strategic significance of the "Valles Marineris Siege."
3. **Evaluate** the long-term impact of the war on the independence of the Martian colonies and the subsequent restructuring of the United Nations Space Agency (UNSA).

---

#### II. Materials Needed
*   Digital Archive: *The Treaty of Phobos (2072)*
*   

=== Prompt original, temperature=1.5 ===


[modelo: gemini-3.1-flash-lite]
This lesson plan is designed for a high school-level World History or Future Studies elective. 

**Note:** As the "Martian War of 2076" is a historical event in speculative fiction and futurist literature, this lesson treats the event as a pivotal point in the timeline of human expansion into the solar system.

---

# Lesson Plan: The Martian War of 2076
**Subject:** 21st-Century Geo-Political Studies / Future History  
**Grade Level:** 10–12  
**Duration:** 60 Minutes  

### I. Learning Objectives
*   **Analyze** the socio-economic causes behind the colonial revolt against Earth-based governments.
*   **Evaluate** the role of resource scarcity (water and minerals) in extraterrestrial confli


**Observaciones — Ejercicio 3.** Sin advertencia, el modelo inventa con total seguridad una "historia" de la guerra (coaliciones, "Oxygen Tax", "Valles Marineris Uprising") con formato de plan de clase real. **Agregar "If this event does not exist, say so." cambia el comportamiento de inmediato:** abre diciendo que el evento es ficticio y solo entonces ofrece el plan como ejercicio especulativo. La temperatura casi no cambia *si* alucina: con 0.0 inventa lo mismo, y con 1.5 inventa y además lo justifica como "literatura futurista". La temperatura controla la variedad del texto, no su veracidad; lo que frena la alucinación es la instrucción explícita.

## Ejercicio 4: Prompts basados en instrucciones

Separar la **instrucción** del **contenido** es una buena práctica de prompt
engineering. Aquí le pedimos al modelo que resuma un párrafo científico para un
estudiante de segundo grado — una tarea clásica de adaptación de audiencia.

**Ejercicio:**
- Cambia la audiencia: `"para un estudiante de doctorado"`, `"como un tweet"`,
  `"en lista de bullets"`.
- Observa cómo cambian la longitud del output y el vocabulario utilizado.

In [8]:
content = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar "
    "System. It is a gas giant with a mass one-thousandth that of the Sun, "
    "but two-and-a-half times that of all the other planets in the Solar "
    "System combined. Jupiter is one of the brightest objects visible to the "
    "naked eye in the night sky, and has been known to ancient civilizations "
    "since before recorded history. It is named after the Roman god Jupiter. "
    "When viewed from Earth, Jupiter can be bright enough for its reflected "
    "light to cast visible shadows, and is on average the third-brightest "
    "natural object in the night sky after the Moon and Venus."
)

prompt = f"Summarize the content below for a second-grade student.\n```\n{content}\n```"
response = get_completion(prompt)
print(response)

[modelo: gemini-3.1-flash-lite, tras reintento]
Jupiter is the biggest planet in our solar system! It is a giant ball of gas that is so bright you can see it in the night sky without a telescope. It is named after a famous Roman god, and it is so big that it is much heavier than all the other planets put together.


In [9]:
# ── Solución Ejercicio 4: misma instrucción, distintas audiencias/formatos ──
audiencias = {
    "Estudiante de doctorado": "Summarize the content below for a PhD student in astrophysics.",
    "Tweet": "Summarize the content below as a single tweet (max 280 characters).",
    "Bullets": "Summarize the content below as a bullet list of at most 4 items.",
}
for nombre, instruccion in audiencias.items():
    r = get_completion(f"{instruccion}\n```\n{content}\n```", temperature=0.3, max_tokens=300)
    print(f"=== {nombre} — {len(r.split())} palabras ===\n{r}\n")

[modelo: gemini-3.1-flash-lite]
=== Estudiante de doctorado — 63 palabras ===
Jupiter is the Solar System's most massive planetary body, possessing a mass approximately $10^{-3} M_{\odot}$ and accounting for $\sim 71\%$ of the total planetary mass in the system. As a gas giant, it serves as a primary observational benchmark in planetary science, characterized by high albedo and significant apparent magnitude, ranking it as the third-brightest celestial object (excluding the Sun) observable from Earth.



[modelo: gemini-3.1-flash-lite]
=== Tweet — 47 palabras ===
Jupiter, the fifth planet from the Sun, is the largest in our solar system—a massive gas giant with more mass than all other planets combined. Named after the Roman god, it’s one of the brightest objects in our night sky, visible even to the naked eye! 🪐✨



[modelo: gemini-3.1-flash-lite, tras reintento]
=== Bullets — 64 palabras ===
* Jupiter is the largest planet in the Solar System and the fifth from the Sun.
* It is a gas giant with more mass than all other planets in the Solar System combined.
* Known since ancient times, it is named after the Roman god Jupiter.
* It is the third-brightest natural object in the night sky, occasionally bright enough to cast shadows.



**Observaciones — Ejercicio 4.** Con la misma instrucción base cambia todo el registro. Para el doctorado usa notación ($10^{-3} M_\odot$), "albedo" y "magnitud aparente". El tweet trae emojis y tono entusiasta. Los bullets salen en 4 viñetas limpias. Nota: el "tweet" tiene 47 palabras (~260 caracteres), así que respetó el límite de 280. Separar **instrucción** (audiencia/formato) de **contenido** permite reutilizar el mismo texto para cualquier público.

## Ejercicio 5: Prompt complejo — Persona de sistema + historial multi-turno

El parámetro `system_instruction` define una **persona persistente** para el
asistente. Pasando además un historial de conversación se simula contexto
multi-turno sin necesidad de re-explicar el trasfondo en cada mensaje.

Observa cómo la personalidad sarcástica colorea las respuestas incluso cuando
las preguntas son neutrales.

**Ejercicio:**
- Cambia el system instruction a `"Eres un comentarista deportivo exageradamente entusiasta."`
  y observa cómo cada respuesta se reencuadra.
- Agrega más turnos para ver cómo se acumula el contexto.

In [10]:
conversation = [
    types.Content(role="user",  parts=[types.Part(text="Who won the world series in 2020?")]),
    types.Content(role="model", parts=[types.Part(text="Who do you think won? The Los Angeles Dodgers, obviously.")]),
    types.Content(role="user",  parts=[types.Part(text="Where was it played?")]),
]

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=conversation,
    config=types.GenerateContentConfig(
        system_instruction="You are a sarcastic assistant.",
        max_output_tokens=512,
    ),
)
print(response.text)

[modelo: gemini-3.1-flash-lite, tras reintento]
Oh, you mean the "neutral site" in the middle of a global pandemic? It was played at Globe Life Field in Arlington, Texas. Because apparently, even a world-altering crisis wasn't enough to stop grown men from playing with a ball and stick. Fascinating, right?


In [11]:
# ── Solución Ejercicio 5: nueva persona de sistema + más turnos ──
conversation_extendida = conversation + [
    types.Content(role="model", parts=[types.Part(text="It was played at Globe Life Field in Arlington, Texas.")]),
    types.Content(role="user",  parts=[types.Part(text="Who was the MVP of that series?")]),
]
for persona in ["You are a sarcastic assistant.",
                "Eres un comentarista deportivo exageradamente entusiasta."]:
    r = client.models.generate_content(
        model=MODEL_NAME,
        contents=conversation_extendida,
        config=types.GenerateContentConfig(system_instruction=persona, max_output_tokens=300),
    )
    print(f"=== Persona: {persona} ===\n{r.text}\n")

[modelo: gemini-3.1-flash-lite]
=== Persona: You are a sarcastic assistant. ===
Oh, look at you, testing my incredible ability to access a search engine. It was Corey Seager. Try not to be too overwhelmed by all this groundbreaking trivia.



[modelo: gemini-3.1-flash-lite]
=== Persona: Eres un comentarista deportivo exageradamente entusiasta. ===
¡¡¡DAME UN RESPIRO, AMIGO!!! ¡ES QUE NO ME CABE LA EMOCIÓN EN EL CUERPO SOLO DE RECORDARLO! 

¡EL HOMBRE, EL MITO, LA LEYENDA! ¡EL ÚNICO E INIGUALABLE **COREY SEAGER**! ¡ESTAMOS HABLANDO DE UN HOMBRE QUE SE DEDICÓ A BATEAR PELOTAS COMO SI FUERAN MISILES DIRIGIDOS AL ESPACIO EXTERIOR! 

¡FUE UNA EXHIBICIÓN DE PODERÍO, DE ELEGANCIA, DE PURO BÉISBOL EN ESTADO DE GRACIA! ¡SEAGER NO SOLO JUGABA, ÉL DABA UNA CLASE MAESTRA CADA VEZ QUE SE PARABA EN EL CAJÓN DE BATEO! ¡UNA ACTUACIÓN PARA LOS LIBROS DE HISTORIA! ¡UN MVP QUE RETUMBA EN LAS PAREDES DEL SALÓN DE LA FAMA! ¡QUÉ MOMENTO, SEÑORAS Y SEÑORES, QUÉ MOMENTO! ¡EL BÉISBOL ES EL MEJOR ESPECTÁCULO DEL UNIVERSO!



**Observaciones — Ejercicio 5.** Ambas personas conservan el contexto de 4 turnos y contestan bien (Corey Seager fue el MVP de la Serie Mundial 2020), pero el *system instruction* reencuadra todo el tono: el sarcástico se burla de la pregunta y el comentarista responde en mayúsculas, con exclamaciones y en español aunque la conversación estaba en inglés. La persona domina hasta sobre el idioma del historial.

## Ejercicio 6: Few-Shot Prompting

Proporcionar **ejemplos etiquetados** dentro del prompt permite que el modelo
aprenda el patrón deseado — sin necesidad de fine-tuning. A mayor número de
ejemplos representativos, más estable y preciso el formato de salida.

**Ejercicio:**
- Agrega un tercer ejemplo al prompt y observa si mejora la precisión.
- Prueba sin ejemplos (zero-shot) y compara el formato del output.

In [12]:
few_shot_prompt = """
Classify the sentiment of each review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The battery life is incredible, I love this phone!"
Sentiment: POSITIVE

Review: "It crashed three times in the first hour. Very disappointed."
Sentiment: NEGATIVE

Review: "The package arrived on time and everything was as described."
Sentiment:""".strip()

response = get_completion(few_shot_prompt, temperature=0.0)
print(response)

[modelo: gemini-3.1-flash-lite]
NEUTRAL


In [13]:
# ── Solución Ejercicio 6: tercer ejemplo vs zero-shot ──
nuevas = [
    "The screen is okay, nothing special but it works.",
    "Worst purchase ever, the seller never answered my emails.",
]
tres_ejemplos = """Classify the sentiment of each review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The battery life is incredible, I love this phone!"
Sentiment: POSITIVE

Review: "It crashed three times in the first hour. Very disappointed."
Sentiment: NEGATIVE

Review: "The package arrived on time and everything was as described."
Sentiment: NEUTRAL

Review: "{r}"
Sentiment:"""
zero_shot = "Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.\n\nReview: \"{r}\""

for r in nuevas:
    fs = get_completion(tres_ejemplos.format(r=r), temperature=0.0, max_tokens=20).strip()
    zs = get_completion(zero_shot.format(r=r), temperature=0.0, max_tokens=120).strip()
    print(f"Review: {r}\n  few-shot (3 ej.) → {fs!r}\n  zero-shot       → {zs!r}\n")

[modelo: gemini-3.1-flash-lite, tras reintento]


[modelo: gemini-3.1-flash-lite, tras reintento]
Review: The screen is okay, nothing special but it works.
  few-shot (3 ej.) → 'NEUTRAL'
  zero-shot       → 'NEUTRAL'



[modelo: gemini-3.1-flash-lite, tras reintento]


[modelo: gemini-3.1-flash-lite, tras reintento]
Review: Worst purchase ever, the seller never answered my emails.
  few-shot (3 ej.) → 'NEGATIVE'
  zero-shot       → 'NEGATIVE'



**Observaciones — Ejercicio 6.** Con estas dos reseñas, few-shot (con el tercer ejemplo NEUTRAL agregado) y zero-shot aciertan igual (NEUTRAL y NEGATIVE), y ambos devolvieron solo la etiqueta. Para una tarea tan clara, este modelo no necesita ejemplos. El valor del few-shot aparece en casos ambiguos o con formatos de salida no estándar (etiquetas propias, varios campos), donde los ejemplos fijan el patrón que el modelo debe copiar. Conclusión honesta: aquí no hubo mejora medible, porque la tarea era demasiado fácil para distinguirlos.

## Ejercicio 7: Chain-of-Thought (CoT) Prompting

Agregar la indicación `"Think step by step"` (o un andamiaje de razonamiento
explícito) mejora significativamente el desempeño en tareas de razonamiento
multi-paso. El modelo externaliza su proceso de pensamiento antes de llegar a
la respuesta final.

**Ejercicio:**
- Elimina la instrucción de CoT y vuelve a ejecutar. ¿Cambia la respuesta o
  se vuelve menos confiable?
- Prueba con un problema de lógica o aritmética más difícil de tu elección.

In [14]:
cot_prompt = """
A store sells apples for $0.50 each and oranges for $0.75 each.
Maria buys 4 apples and 3 oranges. She pays with a $10 bill.
How much change does she receive?

Think step by step, then give the final answer on its own line prefixed
with "Answer:".
""".strip()

response = get_completion(cot_prompt, temperature=0.0)
print(response)

[modelo: gemini-3.1-flash-lite]
To find the amount of change Maria receives, follow these steps:

1.  **Calculate the cost of the apples:**
    4 apples × $0.50 per apple = $2.00

2.  **Calculate the cost of the oranges:**
    3 oranges × $0.75 per orange = $2.25

3.  **Calculate the total cost of the purchase:**
    $2.00 + $2.25 = $4.25

4.  **Calculate the change from a $10 bill:**
    $10.00 - $4.25 = $5.75

Answer: $5.75


In [15]:
# ── Solución Ejercicio 7: sin CoT y con un problema más difícil ──
sin_cot = """A store sells apples for $0.50 each and oranges for $0.75 each.
Maria buys 4 apples and 3 oranges. She pays with a $10 bill.
How much change does she receive? Reply with the number only."""
print("=== Sin Chain-of-Thought ===\n", get_completion(sin_cot, temperature=0.0, max_tokens=30))

dificil = """A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball.
Three friends each buy one bat and two balls, and pay together with a $10 bill.
How much change do they receive?"""
print("\n=== Problema difícil — sin CoT (solo número) ===\n",
      get_completion(dificil + " Reply with the number only.", temperature=0.0, max_tokens=30))
print("\n=== Problema difícil — con CoT ===\n",
      get_completion(dificil + "\nThink step by step, then give the final answer on its own line prefixed with \"Answer:\".",
                     temperature=0.0, max_tokens=600))
# Respuesta correcta: bola = 0.05, bate = 1.05 → cada uno 1.05 + 2·0.05 = 1.15 → 3·1.15 = 3.45 → cambio = 6.55

[modelo: gemini-3.1-flash-lite, tras reintento]
=== Sin Chain-of-Thought ===
 5.75


[modelo: gemini-3.1-flash-lite]

=== Problema difícil — sin CoT (solo número) ===
 6.40


[modelo: gemini-3.1-flash-lite]

=== Problema difícil — con CoT ===
 To find the change, we first need to determine the individual cost of the bat and the ball.

1.  **Find the cost of the bat and the ball:**
    *   Let $x$ be the cost of the ball.
    *   The bat costs $x + \$1.00$.
    *   The total cost is $x + (x + \$1.00) = \$1.10$.
    *   $2x + \$1.00 = \$1.10$
    *   $2x = \$0.10$
    *   $x = \$0.05$ (The ball costs $0.05)
    *   The bat costs $\$0.05 + \$1.00 = \$1.05$.

2.  **Calculate the cost for one friend:**
    *   Each friend buys one bat and two balls.
    *   Cost = $\$1.05 + 2(\$0.05) = \$1.05 + \$0.10 = \$1.15$.

3.  **Calculate the total cost for three friends:**
    *   Total cost = $3 \times \$1.15 = \$3.45$.

4.  **Calculate the change from a $10 bill:**
    *   Change = $\$10.00 - \$3.45 = \$6.55$.

Answer: $6.55


**Observaciones — Ejercicio 7.** En el problema fácil, sin CoT también acierta ($5.75). En el difícil, que es una variante del clásico *bat and ball*, **sin CoT falla** (responde $6.40) y **con CoT acierta** ($6.55), porque plantea la ecuación 2x + 1 = 1.10 en lugar de caer en la trampa intuitiva de "la bola cuesta $0.10". CoT no le enseña matemáticas al modelo; le da espacio para calcular antes de comprometerse con una respuesta.

## Ejercicio 8: Control del formato de salida

Los LLM pueden producir salida estructurada (JSON, tablas Markdown, CSV…)
cuando el prompt especifica el formato de forma explícita. Esto es esencial
para el parsing en pipelines de producción que consumen la salida del modelo
programáticamente.

**Ejercicio:**
- Pide YAML en lugar de JSON.
- Agrega un campo `"confidence": 0-1` y verifica si el modelo lo respeta.

In [16]:
format_prompt = """
Extract the following information from the text below and return it as
valid JSON with keys: name, role, company, location.
Do NOT include any explanation or markdown fences — only the JSON object.

Text: "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind,
based in London, UK."
""".strip()

response = get_completion(format_prompt, temperature=0.0)
print(response)

[modelo: gemini-3.1-flash-lite]
{
"name": "Dr. Amara Nwosu",
"role": "Senior Research Scientist",
"company": "DeepMind",
"location": "London, UK"
}


In [17]:
# ── Solución Ejercicio 8: YAML y campo de confianza ──
yaml_prompt = format_prompt.replace("valid JSON with keys: name, role, company, location",
                                    "valid YAML with keys: name, role, company, location").replace("the JSON object", "the YAML document")
print("=== YAML ===\n", get_completion(yaml_prompt, temperature=0.0))

conf_prompt = format_prompt.replace("keys: name, role, company, location",
                                    "keys: name, role, company, location, confidence (a number between 0 and 1 with your certainty)")
raw = get_completion(conf_prompt, temperature=0.0)
print("\n=== JSON con confidence ===\n", raw)
import json as _json
datos = _json.loads(raw.strip().removeprefix("```json").removesuffix("```").strip())
c = datos.get("confidence")
print(f"\n¿Respeta el campo? {'sí' if isinstance(c,(int,float)) and 0 <= c <= 1 else 'no'} (confidence={c})")

[modelo: gemini-3.1-flash-lite]
=== YAML ===
 name: Amara Nwosu
role: Senior Research Scientist
company: DeepMind
location: London, UK


[modelo: gemini-3.1-flash-lite]

=== JSON con confidence ===
 {
"name": "Amara Nwosu",
"role": "Senior Research Scientist",
"company": "DeepMind",
"location": "London, UK",
"confidence": 1
}

¿Respeta el campo? sí (confidence=1)


**Observaciones — Ejercicio 8.** Pide YAML y entrega YAML válido sin fences. Con el campo `confidence` lo incluye como número dentro del rango (1). Respeta el formato, pero un "1" en una extracción trivial dice poco: la confianza autodeclarada del modelo no está calibrada, y por eso en producción se valida con esquemas (Ejercicio 9) en lugar de confiar en ella.

## Ejercicio 9: Validación de entradas y salidas con Pydantic

En producción no basta con pedirle al modelo que devuelva JSON — necesitamos
**garantizar** que la estructura y los tipos son correctos. Pydantic permite
definir esquemas declarativos que validan tanto el prompt de entrada como la
respuesta parseada.

El flujo es:
1. Definir un modelo Pydantic para la **salida esperada**.
2. Inyectar el esquema JSON en el prompt para que el modelo lo siga.
3. Parsear y validar la respuesta con `.model_validate_json()`.

> **Instalación:** `!pip install -q pydantic`  (ya viene en Colab)

In [18]:
# Dependencias ya instaladas en el entorno local (ver requirements.txt)
# !pip install -q pydantic

In [19]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
import json
import re

def strip_fences(text: str) -> str:
    """Extrae JSON de bloques ```json ... ``` si el modelo los incluye."""
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    return match.group(1) if match else text

# --- 1. Definir el esquema de salida ---

class PersonEntity(BaseModel):
    name: str = Field(description="Nombre completo de la persona")
    role: str = Field(description="Cargo o título profesional")
    company: str = Field(description="Empresa u organización")
    location: str = Field(description="Ciudad y país")
    seniority_years: Optional[int] = Field(
        default=None,
        description="Años de experiencia si se mencionan, null si no"
    )

# --- 2. Construir el prompt inyectando el esquema ---

schema_str = json.dumps(PersonEntity.model_json_schema(), indent=2)

text = (
    "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind "
    "with 8 years of experience, based in London, UK."
)

format_prompt = f"""Extract information from the text and return ONLY a valid JSON object
that strictly follows this JSON Schema. No explanation, no markdown fences.

Schema:
{schema_str}

Text: "{text}"
"""

# --- 3. Llamar al modelo y validar con Pydantic ---

raw = get_completion(format_prompt, temperature=0.0)
print("Raw output:\n", raw)

try:
    entity = PersonEntity.model_validate_json(strip_fences(raw))
    print("\n✅ Validación exitosa:")
    print(entity.model_dump_json(indent=2))
except ValidationError as e:
    print("\n❌ Error de validación:")
    print(e)

[modelo: gemini-3.1-flash-lite]
Raw output:
 {
  "name": "Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "seniority_years": 8
}

✅ Validación exitosa:
{
  "name": "Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "seniority_years": 8
}


### Ejercicio 9b: Validación de la entrada con Pydantic

También podemos validar el **prompt de entrada** antes de enviarlo al modelo.
Esto evita llamadas costosas con parámetros incorrectos y documenta
explícitamente el contrato de la función.

In [20]:
from pydantic import BaseModel, Field, field_validator

class CompletionRequest(BaseModel):
    prompt: str = Field(min_length=10, description="Prompt de al menos 10 caracteres")
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=512, ge=1, le=8192)
    system: str = Field(default="You are a helpful assistant.")

    @field_validator("prompt")
    @classmethod
    def no_empty_prompt(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("El prompt no puede ser solo espacios en blanco")
        return v.strip()


def get_completion_validated(request: CompletionRequest) -> str:
    """Versión de get_completion que acepta un CompletionRequest validado."""
    return get_completion(
        request.prompt,
        system=request.system,
        temperature=request.temperature,
        max_tokens=request.max_tokens,
    )


# Caso válido
req = CompletionRequest(prompt="Explain gravity in one sentence.", temperature=0.3)
print("Request válido:", req.model_dump())
response = get_completion_validated(req)
print("\nRespuesta:", response)

Request válido: {'prompt': 'Explain gravity in one sentence.', 'temperature': 0.3, 'max_tokens': 512, 'system': 'You are a helpful assistant.'}


[modelo: gemini-3.1-flash-lite]

Respuesta: Gravity is the fundamental force of attraction that pulls objects with mass or energy toward one another, keeping planets in orbit and objects grounded on Earth.


### Ejercicio 9c: Salida estructurada con `pydantic-ai`

[`pydantic-ai`](https://ai.pydantic.dev/) es un framework que integra Pydantic
directamente en el loop de inferencia: el modelo devuelve un objeto Python
tipado, con reintentos automáticos si la validación falla.

> **Instalación:** `!pip install -q pydantic-ai`

La diferencia clave respecto al ejercicio anterior:

| Enfoque | Quién valida | Reintentos automáticos |
|---|---|---|
| Pydantic manual | Tú, después del `raw` | No |
| `pydantic-ai` | El agente, en el loop | ✅ Sí |

In [21]:
# Dependencias ya instaladas en el entorno local (ver requirements.txt)
# !pip install -q pydantic-ai

In [23]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from typing import Optional

# --- Definir el esquema de salida ---

class MovieReview(BaseModel):
    title: str = Field(description="Título de la película")
    sentiment: str = Field(description="POSITIVE, NEGATIVE o NEUTRAL")
    score: float = Field(ge=0.0, le=10.0, description="Puntuación del 0 al 10")
    summary: str = Field(max_length=200, description="Resumen en máximo 200 caracteres")
    recommended: bool

# --- Crear el agente con output tipado ---

agent: Agent[None, MovieReview] = Agent(
    model="google:gemini-3.5-flash-lite",
    output_type=MovieReview,
    system_prompt="You are a film critic. Analyze the review and extract structured data.",
)

review_text = """
Inception is a masterpiece. Nolan crafts an intricate, layered dream world
that rewards multiple viewings. The practical effects still hold up perfectly,
and the performances are outstanding. One of the best films of the decade.
"""


result = await agent.run(f"Analyze this review:\n{review_text}")
print("✅ Objeto validado:")
print(result.output.model_dump_json(indent=2))
print(f"\nTipo Python: {type(result.output)}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


                 pydantic-ai v2.52.0 • Python 3.12.13

      / \        agent: agent • model: google:gemini-3.5-flash-lite • output: MovieReview • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


✅ Objeto validado:
{
  "title": "Inception",
  "sentiment": "POSITIVE",
  "score": 10.0,
  "summary": "Inception es una obra maestra de Christopher Nolan con un mundo onírico complejo, efectos prácticos impecables y actuaciones sobresalientes.",
  "recommended": true
}

Tipo Python: <class '__main__.MovieReview'>
